# 01 - Deploy Infrastructure

This notebook deploys `infra/main.bicep` and writes the core deployment outputs into `env/.env` so later notebooks can run without manual value copy/paste.

**Estimated time:** 10-20 minutes

---

## What gets deployed

- Azure AI Services account (`kind: AIServices`)
- Foundry project (`Microsoft.CognitiveServices/accounts/projects`)
- Azure Function App + App Insights for later function publishing

---

## Step 1 - Set deployment variables

In [ ]:
RESOURCE_GROUP = "rg-speech01-foundry"
LOCATION = "eastus2"
DEMO_ID = "speech01"

print(f"Resource group : {RESOURCE_GROUP}")
print(f"Location       : {LOCATION}")
print(f"Demo ID        : {DEMO_ID}")


## Step 2 - Create the resource group

In [ ]:
!az group create \
    --name {RESOURCE_GROUP} \
    --location {LOCATION} \
    --output table

## Step 3 - Deploy Bicep template

In [ ]:
import re
import subprocess
import sys
from pathlib import Path

demo_root = Path("..").resolve()
if str(demo_root) not in sys.path:
    sys.path.insert(0, str(demo_root))

from app.notebook_helpers import resolve_az_cli


def run_deployment(az_cmd: str, include_deployer_principal: bool, deployer_object_id: str):
    deploy_command = [
        az_cmd,
        "deployment",
        "group",
        "create",
        "--resource-group",
        RESOURCE_GROUP,
        "--template-file",
        "../infra/main.bicep",
        "--parameters",
        f"location={LOCATION}",
        f"demoId={DEMO_ID}",
    ]
    if include_deployer_principal and deployer_object_id:
        deploy_command.append(f"deployerPrincipalId={deployer_object_id}")
    deploy_command.extend(["--output", "table"])

    return subprocess.run(deploy_command, capture_output=True, text=True)


def extract_soft_deleted_account_name(stderr_text: str) -> str:
    # Example path in error: .../Microsoft.CognitiveServices/accounts/<name>'
    match = re.search(r"/Microsoft\.CognitiveServices/accounts/([^'\s\"]+)", stderr_text)
    return match.group(1) if match else ""


def purge_soft_deleted_account(az_cmd: str, account_name: str) -> subprocess.CompletedProcess:
    return subprocess.run(
        [
            az_cmd,
            "cognitiveservices",
            "account",
            "purge",
            "--name",
            account_name,
            "--location",
            LOCATION,
            "--resource-group",
            RESOURCE_GROUP,
        ],
        capture_output=True,
        text=True,
        check=False,
    )


az_cmd = resolve_az_cli()
if not az_cmd:
    raise RuntimeError("Azure CLI not found. Install it or add it to PATH.")

deployer_object_id_result = subprocess.run(
    [az_cmd, "ad", "signed-in-user", "show", "--query", "id", "-o", "tsv"],
    capture_output=True,
    text=True,
)
deployer_object_id = deployer_object_id_result.stdout.strip()

if deployer_object_id:
    print(f"Using deployer principal ID for storage upload RBAC: {deployer_object_id}")
else:
    print(
        "Could not detect signed-in user object ID automatically; continuing without "
        "deployerPrincipalId."
    )

include_deployer_principal = bool(deployer_object_id)
retried_role_assignment = False
retried_after_purge = False

while True:
    # Deploy the core infrastructure for the prototype.
    redeploy_result = run_deployment(
        az_cmd=az_cmd,
        include_deployer_principal=include_deployer_principal,
        deployer_object_id=deployer_object_id,
    )
    print(redeploy_result.stdout)

    if redeploy_result.returncode == 0:
        break

    stderr_text = redeploy_result.stderr.strip()

    if (
        "RoleAssignmentExists" in stderr_text
        and include_deployer_principal
        and not retried_role_assignment
    ):
        print(
            "Detected existing storage RBAC assignment with a different role-assignment ID. "
            "Retrying deployment without deployerPrincipalId."
        )
        include_deployer_principal = False
        retried_role_assignment = True
        continue

    if "FlagMustBeSetForRestore" in stderr_text and not retried_after_purge:
        soft_deleted_name = extract_soft_deleted_account_name(stderr_text)
        if not soft_deleted_name:
            raise RuntimeError(
                "Bicep deployment hit a soft-deleted AI Services account, but the account name "
                "could not be parsed from Azure CLI error output.\n"
                f"stderr: {stderr_text}\n"
                f"stdout: {redeploy_result.stdout.strip()}"
            )

        print(
            f"Detected soft-deleted AI Services account '{soft_deleted_name}'. "
            "Attempting purge, then retrying deployment once."
        )
        purge_result = purge_soft_deleted_account(az_cmd=az_cmd, account_name=soft_deleted_name)
        if purge_result.returncode != 0:
            raise RuntimeError(
                f"Failed to purge soft-deleted account '{soft_deleted_name}'.\n"
                f"stderr: {purge_result.stderr.strip()}\n"
                f"stdout: {purge_result.stdout.strip()}"
            )

        print(f"Purged soft-deleted account '{soft_deleted_name}'. Retrying deployment...")
        retried_after_purge = True
        continue

    raise RuntimeError(
        f"Bicep deployment failed.\n"
        f"stderr: {redeploy_result.stderr.strip()}\n"
        f"stdout: {redeploy_result.stdout.strip()}"
    )

## Step 4 - Retrieve deployment outputs

Query the latest deployment in this resource group and capture the values needed for the later notebooks.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

demo_root = Path("..").resolve()
if str(demo_root) not in sys.path:
    sys.path.insert(0, str(demo_root))

from app.notebook_helpers import resolve_az_cli


def run_az_json(command: list[str], context: str):
    result = subprocess.run(
        command,
        capture_output=True,
        text=True,
        check=False,
    )
    if result.returncode != 0:
        raise RuntimeError(
            f"Azure CLI command failed while {context}.\n"
            f"command: {' '.join(command)}\n"
            f"stderr: {result.stderr.strip()}\n"
            f"stdout: {result.stdout.strip()}"
        )

    payload = result.stdout.strip()
    if not payload:
        return None

    try:
        return json.loads(payload)
    except json.JSONDecodeError as ex:
        raise RuntimeError(
            f"Azure CLI returned non-JSON output while {context}.\n"
            f"command: {' '.join(command)}\n"
            f"stdout: {payload}\n"
            f"stderr: {result.stderr.strip()}"
        ) from ex


az_cmd = resolve_az_cli()
if not az_cmd:
    raise RuntimeError("Azure CLI not found from this kernel. Install it or add it to PATH.")

# First try latest deployment outputs in this resource group.
outputs = run_az_json(
    [
        az_cmd,
        "deployment",
        "group",
        "list",
        "--resource-group",
        RESOURCE_GROUP,
        "--query",
        "[0].properties.outputs",
        "--output",
        "json",
    ],
    context="retrieving latest deployment outputs",
)

# Fallback to explicit deployment name used by this notebook flow.
if not outputs:
    outputs = run_az_json(
        [
            az_cmd,
            "deployment",
            "group",
            "show",
            "--resource-group",
            RESOURCE_GROUP,
            "--name",
            "main",
            "--query",
            "properties.outputs",
            "--output",
            "json",
        ],
        context="retrieving deployment outputs from deployment 'main'",
    )

if not outputs:
    raise RuntimeError(
        "No deployment outputs were found in the resource group. "
        "Rerun Step 3 first, then rerun this cell."
    )

ai_account_name = outputs.get("aiServicesAccountName", {}).get("value")
ai_endpoint = outputs.get("aiServicesEndpoint", {}).get("value")
auth_mode = "aad"
function_app_name = outputs.get("functionAppName", {}).get("value", "")
function_url = outputs.get("functionUrl", {}).get("value", "")
app_insights_name = outputs.get("appInsightsName", {}).get("value", "")

if not ai_account_name or not ai_endpoint:
    raise RuntimeError(
        "Deployment outputs are missing required values (aiServicesAccountName/aiServicesEndpoint). "
        "Rerun Step 3 and check deployment success."
    )

print(f"AI account             : {ai_account_name}")
print(f"AI endpoint            : {ai_endpoint}")
print(f"Auth mode              : {auth_mode}")
print(f"Function app           : {function_app_name or '(not yet deployed)'}")
print(f"Function URL           : {function_url or '(not yet available)'}")
print(f"App Insights           : {app_insights_name or '(not yet deployed)'}")

## Step 5 - Write outputs to `env/.env`

This persists values from Step 4 for the configure and test notebooks.

> **Security note:** `env/.env` is git-ignored. Never commit it.

In [ ]:
import os
import pathlib
import subprocess

subscription_id = os.getenv("AZURE_SUBSCRIPTION_ID", "").strip()
if not subscription_id:
    sub_result = subprocess.run(
        [az_cmd, "account", "show", "--query", "id", "-o", "tsv"],
        capture_output=True,
        text=True,
        check=True,
    )
    subscription_id = sub_result.stdout.strip()

if not subscription_id:
    raise RuntimeError(
        "Could not resolve AZURE_SUBSCRIPTION_ID from Azure CLI. "
        "Run Step 4 first and confirm Azure CLI authentication."
    )

env_lines = [
    "# Auto-generated by 01_deploy_infra.ipynb - do not commit",
    f"AZURE_SUBSCRIPTION_ID={subscription_id}",
    f"AZURE_RESOURCE_GROUP={RESOURCE_GROUP}",
    f"AZURE_AI_ENDPOINT={ai_endpoint}",
    f"AZURE_AUTH_MODE={auth_mode}",
    "AZURE_SPEECH_VOICE=en-US-AvaMultilingualNeural",
    f"AZURE_FUNCTION_APP_NAME={function_app_name}",
    f"AZURE_FUNCTION_URL={function_url}",
    f"AZURE_APP_INSIGHTS_NAME={app_insights_name}",
    "AZURE_FUNCTION_KEY=",
]

env_content = "\n".join(env_lines) + "\n"

env_path = pathlib.Path("../env/.env")
env_path.write_text(env_content, encoding="utf-8")
print(f"✅ Written to {env_path.resolve()}")

## Step 5 - Verify `env/.env`

Confirm the generated environment file exists and contains the bootstrap values needed by the later notebooks.


In [ ]:
import pathlib

env_path = pathlib.Path("../env/.env")
example_path = pathlib.Path("../env/.env.example")

if env_path.exists():
    print("env/.env found")
    print(f"Path: {env_path.resolve()}")
    env_text = env_path.read_text(encoding="utf-8")
    if "AZURE_FUNCTION_KEY=" in env_text:
        print("Notebook 2 will populate AZURE_FUNCTION_KEY after the Function App code is published.")
else:
    print("ERROR: env/.env not found")
    print("Step 5 should generate this file.")
    print(f"Expected variable contract from {example_path}:")
    print(example_path.read_text(encoding="utf-8"))
    raise FileNotFoundError(env_path)

---

## ✅ Infrastructure deployed

Move to **`02_configure.ipynb`** to validate Speech readiness and inspect voices.

---

## Tear-down

In [ ]:
# Uncomment and run to delete all resources when finished.
# !az group delete --name {RESOURCE_GROUP} --yes --no-wait
# print("Resource group deletion triggered.")